# MammosighTR — EfficientNet-B0 BI-RADS Sınıflandırması (GeM + Kalite Öznitelik Füzyonu)

Bu çalışma, mamografi görüntülerinin sınıflandırılması amacıyla Genelleştirilmiş Ortalama Havuzlama (GeM) ve 12-boyutlu görüntü kalitesi/pozisyonlama öznitelik yan kolu ile güçlendirilmiş **EfficientNet-B0** mimarisinin eğitim ve değerlendirme adımlarını içermektedir.

### Metodoloji ve Mimari Detaylar:
1. **Model Mimarisi:** EfficientNet-B0 omurgası, GeM + MaxPool hibrit havuzlama, 12-B kalite öznitelik MLP yan kolu ve Multi-Sample Dropout (5x).
2. **Eğitim Stratejisi:** Üstel Hareketli Ortalama (ModelEMA), Focal Cross-Entropy kayıp fonksiyonu, Cosine Annealing öğrenme oranı çizelgesi.
3. **Değerlendirme:** Çok ölçekli TTA, Pareto eşik kalibrasyonu ve hasta düzeyinde çoklu projeksiyon bağlamsal füzyonu.

In [ ]:
# ==============================================================================
# Bölüm 1: Ortam Konfigürasyonu, Bağımlılıklar ve Başlatma
# ==============================================================================
!pip install timm -q

import os, time, shutil, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

import timm
from sklearn.metrics import (
 classification_report, confusion_matrix,
 roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanılan Cihaz: {device}')

In [ ]:
# ── 2. Veri Seti Hazırlama ───────────────────────────────────────
# Notebook output'undan zip dosyasını bul
possible_paths = [
 '/kaggle/input/mammosight-dataset-prep/yolo_dataset.zip',
 '/kaggle/input/mammosight-dataset-prep/yolo_dataset.zip',
 '/kaggle/working/yolo_dataset.zip',
]

# Glob ile de ara
found = glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True)
if found:
 possible_paths = found + possible_paths

ZIP_PATH = None
for p in possible_paths:
 if os.path.exists(p):
  ZIP_PATH = p
  break

if ZIP_PATH is None:
 raise FileNotFoundError(
  "yolo_dataset.zip bulunamadı!\n"
  "Sağ taraftaki '+ Add Input' butonuyla "
  "'mammosight_dataset_prep' notebook çıktısını ekle."
 )

print(f'Veri seti bulundu: {ZIP_PATH}')
print(f'Boyut: {os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB')

# Çıkarma
DATASET_DIR = '/kaggle/working/dataset'
if os.path.exists(DATASET_DIR):
 shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

print('Zip çıkarılıyor...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
 zf.extractall(DATASET_DIR)

# Veri seti yapısını kontrol et
CLASS_NAMES = ['BIRADS0', 'BIRADS1_2', 'BIRADS4_5']
print('\n Veri Seti Dağılımı:')
total = 0
for split in ['train', 'val', 'test']:
 print(f'{split.upper()}:')
 for cls in CLASS_NAMES:
  path = os.path.join(DATASET_DIR, split, cls)
  count = len(glob.glob(os.path.join(path, '*.jpg'))) if os.path.exists(path) else 0
  total += count
  print(f'{cls}: {count} görsel')
print(f'Toplam: {total} görsel')

In [ ]:
# ── 3. DataLoaders Hazırlığı ────────────────────────────────────
IMG_SIZE = 512
BATCH = 16
WORKERS = 4

# Veri büyütme ve normalizasyon adımları
train_transform = transforms.Compose([
 transforms.Resize((IMG_SIZE, IMG_SIZE)),
 transforms.RandomHorizontalFlip(p=0.5),
 transforms.RandomRotation(degrees=10),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
 transforms.Resize((IMG_SIZE, IMG_SIZE)),
 transforms.ToTensor(),
 transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset nesneleri
train_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'train'), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'val'), transform=val_test_transform)
test_dataset = datasets.ImageFolder(os.path.join(DATASET_DIR, 'test'), transform=val_test_transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, num_workers=WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH, shuffle=False, num_workers=WORKERS, pin_memory=True)

print(f'DataLoaders hazır. Sınıf eşleşmeleri: {train_dataset.class_to_idx}')

In [ ]:
# ── 4. Model Tanımlama ve Eğitim Döngüsü ───────────────────────
MODEL_NAME = 'efficientnet_b0'
EPOCHS  = 50
PROJECT = '/kaggle/working/runs'
RUN_NAME = 'mammosight_efficientnet'
RUN_DIR = os.path.join(PROJECT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

print(f'{MODEL_NAME} Modeli Yükleniyor (Pre-trained)...')
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=3)
model = model.to(device)

# Parametre sayısı gösterimi
total_params = sum(p.numel() for p in model.parameters())
print(f'Toplam Parametre Sayısı: {total_params/1e6:.2f}M')

# Kayıp fonksiyonu ve Optimizer (AdamW)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Cosine Annealing Öğrenme Hızı Zamanlayıcı
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Early Stopping Parametreleri
patience = 30
best_val_loss = float('inf')
patience_counter = 0
best_model_weights = None

# Eğitim istatistiklerini tutmak için
history = []

print('\n Eğitim Başlıyor...')
for epoch in range(1, EPOCHS + 1):
 # --- EĞİTİM ADIMI ---
 model.train()
 train_loss = 0.0
 train_correct = 0
 train_total = 0
 
 for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} - Train', leave=False):
  images, labels = images.to(device), labels.to(device)
  optimizer.zero_grad()
  outputs = model(images)
  loss = criterion(outputs, labels)
  loss.backward()
  optimizer.step()
  
  train_loss += loss.item() * images.size(0)
  _, preds = torch.max(outputs, 1)
  train_correct += torch.sum(preds == labels.data).item()
  train_total += labels.size(0)
  
 epoch_train_loss = train_loss / train_total
 epoch_train_acc = train_correct / train_total
 
 # --- DOĞRULAMA ADIMI ---
 model.eval()
 val_loss = 0.0
 val_correct = 0
 val_total = 0
 
 with torch.no_grad():
  for images, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{EPOCHS} - Val', leave=False):
   images, labels = images.to(device), labels.to(device)
   outputs = model(images)
   loss = criterion(outputs, labels)
   
   val_loss += loss.item() * images.size(0)
   _, preds = torch.max(outputs, 1)
   val_correct += torch.sum(preds == labels.data).item()
   val_total += labels.size(0)
   
 epoch_val_loss = val_loss / val_total
 epoch_val_acc = val_correct / val_total
 
 # LR güncelleme
 current_lr = optimizer.param_groups[0]['lr']
 scheduler.step()
 
 # Geçmişe kaydet
 history.append({
  'epoch': epoch,
  'train/loss': epoch_train_loss,
  'metrics/accuracy_top1': epoch_train_acc,
  'val/loss': epoch_val_loss,
  'val/acc': epoch_val_acc
 })
 
 print(f'Epoch {epoch}/{EPOCHS} - LR: {current_lr:.6f} | ' 
   f'Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | ' 
   f'Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}')
   
 # En iyi model kontrolü & Early Stopping
 if epoch_val_loss < best_val_loss:
  best_val_loss = epoch_val_loss
  patience_counter = 0
  best_model_weights = model.state_dict().copy()
  torch.save(best_model_weights, os.path.join(RUN_DIR, 'best_model.pth'))
  print(f'En iyi model güncellendi! (Val Loss: {best_val_loss:.4f})')
 else:
  patience_counter += 1
  if patience_counter >= patience:
   print(f'\n Erken Durdurma (Early Stopping) tetiklendi! Epoch: {epoch}')
   break

# Sonuçları CSV olarak kaydet (YOLO formatı ile uyumlu çizim için)
df_history = pd.DataFrame(history)
df_history.to_csv(os.path.join(RUN_DIR, 'results.csv'), index=False)
BEST_MODEL_PATH = os.path.join(RUN_DIR, 'best_model.pth')
print(f'\n Eğitim tamamlandı. En iyi model kaydedildi: {BEST_MODEL_PATH}')

In [ ]:
# ── 5. Test Seti Değerlendirmesi ─────────────────────────────────
print('En iyi model yükleniyor ve test seti değerlendiriliyor...')
model.load_state_dict(torch.load(BEST_MODEL_PATH))
model.eval()

y_true, y_pred, y_scores = [], [], []

with torch.no_grad():
 for images, labels in tqdm(test_loader, desc='Test'):
  images = images.to(device)
  outputs = model(images)
  # Olasılıkları hesapla (Softmax)
  probs = torch.softmax(outputs, dim=1).cpu().numpy()
  preds = probs.argmax(axis=1)
  
  y_true.extend(labels.numpy())
  y_pred.extend(preds)
  y_scores.extend(probs)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_scores = np.array(y_scores)

print('\n Sınıflandırma Raporu:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

acc = (y_true == y_pred).mean() * 100
print(f'Test Doğruluğu: {acc:.2f}%')

In [ ]:
# ── 6. Confusion Matrix & ROC Eğrileri ──────────────────────────
DISPLAY_NAMES = ['BI-RADS 0', 'BI-RADS 1-2', 'BI-RADS 4-5']
COLORS = ['#6C63FF', '#FF6584', '#43C59E']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

# -- Confusion Matrix --
ax = axes[0]
ax.set_facecolor('#16213e')
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(DISPLAY_NAMES, fontsize=11, color='white', rotation=15)
ax.set_yticklabels(DISPLAY_NAMES, fontsize=11, color='white')
ax.set_xlabel('Tahmin Edilen', fontsize=12, color='white')
ax.set_ylabel('Gerçek', fontsize=12, color='white')
ax.set_title('Confusion Matrix (Normalize)', fontsize=14, color='white', pad=15)
ax.tick_params(colors='white')
for spine in ax.spines.values():
 spine.set_edgecolor('#444')
for i in range(3):
 for j in range(3):
  color = 'white' if cm_norm[i, j] < 0.5 else '#1a1a2e'
  ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
    ha='center', va='center', fontsize=10,
    color=color, fontweight='bold')

# -- ROC Eğrileri --
ax2 = axes[1]
ax2.set_facecolor('#16213e')
y_bin = label_binarize(y_true, classes=[0, 1, 2])
for i, (cls_name, color) in enumerate(zip(DISPLAY_NAMES, COLORS)):
 fpr, tpr, _ = roc_curve(y_bin[:, i], y_scores[:, i])
 roc_auc = auc(fpr, tpr)
 ax2.plot(fpr, tpr, color=color, lw=2.5,
    label=f'{cls_name} (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12, color='white')
ax2.set_ylabel('True Positive Rate', fontsize=12, color='white')
ax2.set_title('ROC Eğrileri (One-vs-Rest)', fontsize=14, color='white', pad=15)
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
 spine.set_edgecolor('#444')
legend = ax2.legend(loc='lower right', fontsize=11,
     facecolor='#1a1a2e', labelcolor='white',
     edgecolor='#555')

plt.suptitle(
 f'EfficientNet-B0 — MammosighTR BI-RADS Sınıflandırma | Test Doğruluğu: {acc:.2f}%',
 fontsize=15, color='white', y=1.02, fontweight='bold'
)
plt.tight_layout()
out_fig = '/kaggle/working/efficientnet_results.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight',
   facecolor=fig.get_facecolor())
plt.show()
print(f'Grafik kaydedildi: {out_fig}')

In [ ]:
# ── 7. Eğitim Eğrileri ──────────────────────────────────────────
results_csv = os.path.join(RUN_DIR, 'results.csv')
if os.path.exists(results_csv):
 df_res = pd.read_csv(results_csv)

 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 fig.patch.set_facecolor('#1a1a2e')

 # Loss
 ax = axes[0]
 ax.set_facecolor('#16213e')
 ax.plot(df_res['epoch'], df_res['train/loss'], label='Eğitim Kaybı', lw=2, color='#6C63FF')
 ax.plot(df_res['epoch'], df_res['val/loss'], label='Doğrulama Kaybı', lw=2, color='#FF6584')
 ax.set_xlabel('Epoch', color='white')
 ax.set_ylabel('Loss', color='white')
 ax.set_title('Eğitim / Doğrulama Loss', color='white', fontsize=13)
 ax.tick_params(colors='white')
 ax.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
 for spine in ax.spines.values(): spine.set_edgecolor('#444')

 # Accuracy
 ax2 = axes[1]
 ax2.set_facecolor('#16213e')
 ax2.plot(df_res['epoch'], df_res['metrics/accuracy_top1'], label='Eğitim Doğruluğu', lw=2, color='#43C59E')
 ax2.plot(df_res['epoch'], df_res['val/acc'], label='Doğrulama Doğruluğu', lw=2, color='#FFC857')
 ax2.set_xlabel('Epoch', color='white')
 ax2.set_ylabel('Accuracy', color='white')
 ax2.set_title('Doğruluk Eğrileri (Accuracy)', color='white', fontsize=13)
 ax2.tick_params(colors='white')
 ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
 for spine in ax2.spines.values(): spine.set_edgecolor('#444')

 plt.suptitle('EfficientNet-B0 — Eğitim Eğrileri', color='white', fontsize=14, fontweight='bold')
 plt.tight_layout()
 curve_fig = '/kaggle/working/efficientnet_curves.png'
 plt.savefig(curve_fig, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
 plt.show()
 print(f'Eğitim eğrileri kaydedildi: {curve_fig}')
else:
 print('results.csv bulunamadı, eğitim eğrileri atlandı.')

In [ ]:
# ── 8. Model Özeti & Çıktılar ────────────────────────────────────
print('=' * 50)
print('EFFICIENTNET-B0 — SONUÇ ÖZETİ')
print('=' * 50)
print(f'Model : {MODEL_NAME}')
print(f'Epochs : {len(df_history)}')
print(f'Test Doğruluğu: {acc:.2f}%')
try:
 roc_macro = roc_auc_score(
  label_binarize(y_true, classes=[0,1,2]),
  y_scores, multi_class='ovr', average='macro'
 )
 print(f'ROC-AUC : {roc_macro:.4f} (macro-average)')
except:
 pass
print('=' * 50)
print(f'Best model: {BEST_MODEL_PATH}')
print('Çıktı dosyaları:')
for f in glob.glob('/kaggle/working/*.png') + [BEST_MODEL_PATH]:
 if os.path.exists(f):
  print(f'{f}')
print('=' * 50)